<a href="https://colab.research.google.com/github/Tomas-Turner/Unit2_Flight_Team10/blob/individual/Unit2_Jack_BQML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Unit 2 — Team Classification (Flights, BQML)

**Goal (team):** Build an *ops-ready* classifier in **BigQuery ML** to predict **`diverted`** on U.S. flights. Minimal handholding by design.

**What you deliver (inside this notebook):**
- One **LOGISTIC_REG** model (baseline), one **engineered** model using `TRANSFORM`
- **Evaluation** via `ML.EVALUATE` and **confusion matrices** (default 0.5 + your custom threshold)
- **Threshold choice** + 3–5 sentence ops justification
- Embedded **rubric** below (self-check before submission)

> Choose *one* dataset table that exists at your institution:  
> • `bigquery-public-data.faa.us_flights` **or** `bigquery-public-data.flights.*`  
> Make sure the table has `carrier`, `dep_delay`, `arr_delay` (for filters), `origin`, `dest`, `diverted` (or equivalent).


In [5]:
# Install Kaggle CLI (if needed) and place kaggle.json
!pip -q install kaggle

from google.colab import files
print("👉 Upload your kaggle.json (Kaggle > Account > Create New API Token)")
_ = files.upload()  # pick kaggle.json from your machine

!mkdir -p ~/.kaggle
!mv -f kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle API ready")


👉 Upload your kaggle.json (Kaggle > Account > Create New API Token)


Saving kaggle.json to kaggle.json
✅ Kaggle API ready


In [6]:
# Download the exact dataset
!kaggle datasets download -d mexwell/carrier-on-time-performance-dataset -p ./kaggle_carrier

# Unzip to a folder
!unzip -o ./kaggle_carrier/carrier-on-time-performance-dataset.zip -d ./kaggle_carrier/data

# (Optional) peek at what files we got
!ls -lah ./kaggle_carrier/data | sed -n '1,120p'


Dataset URL: https://www.kaggle.com/datasets/mexwell/carrier-on-time-performance-dataset
License(s): Community Data License Agreement - Sharing - Version 1.0
 89% 139M/156M [00:00<00:00, 1.45GB/s]
100% 156M/156M [00:00<00:00, 1.33GB/s]
Archive:  ./kaggle_carrier/carrier-on-time-performance-dataset.zip
  inflating: ./kaggle_carrier/data/airline_2m.csv  
total 842M
drwxr-xr-x 2 root root 4.0K Nov 11 22:28 .
drwxr-xr-x 3 root root 4.0K Nov 11 22:28 ..
-rw-r--r-- 1 root root 842M Aug 11  2023 airline_2m.csv


In [7]:
from google.colab import auth
auth.authenticate_user()

import time, subprocess, json
from google.cloud import bigquery

PROJECT_ID = "mgmt-467-94721"   # <-- your project
REGION     = "US"

# create a globally-unique bucket name
BUCKET_NAME = f"{PROJECT_ID}-kaggle-{int(time.time())}"
print("Bucket:", BUCKET_NAME)

# make the bucket (ignore error if it already exists)
!gsutil mb -p {PROJECT_ID} -l {REGION} gs://{BUCKET_NAME} || true

# copy ALL data files (CSV/Parquet/etc.) up to GCS under a fixed prefix
!gsutil -m cp -r ./kaggle_carrier/data/* gs://{BUCKET_NAME}/carrier/
print("✅ Uploaded files to gs://{BUCKET_NAME}/carrier/")


Bucket: mgmt-467-94721-kaggle-1762900137
Creating gs://mgmt-467-94721-kaggle-1762900137/...
Copying file://./kaggle_carrier/data/airline_2m.csv [Content-Type=text/csv]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

\ [1/1 files][841.3 MiB/841.3 MiB] 100% Done  72.3 MiB/s ETA 00:00:00           
Operation completed over 1 objects/841.3 MiB.                           

In [8]:
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID)

DATASET = "kaggle_flights"
TABLE   = "carrier_raw"
table_id = f"{PROJECT_ID}.{DATASET}.{TABLE}"

# Create dataset if needed
bq.query(f"CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}`", location=REGION).result()

# Try CSV first; if your files are parquet, flip to PARQUET block below.
uri_csv = f"gs://{BUCKET_NAME}/carrier/*.csv"

job_config = bigquery.LoadJobConfig(
    autodetect=True,
    skip_leading_rows=1,
    source_format=bigquery.SourceFormat.CSV,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    allow_quoted_newlines=True,
    allow_jagged_rows=True,
)

load_job = bq.load_table_from_uri(uri_csv, table_id, job_config=job_config, location=REGION)
try:
    load_job.result()
    print("✅ Loaded CSVs into:", table_id)
except Exception as e:
    print("CSV load failed, trying PARQUET…", e)
    # ---- If dataset is parquet, use this fallback ----
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    uri_parquet = f"gs://{BUCKET_NAME}/carrier/*.parquet"
    load_job = bq.load_table_from_uri(uri_parquet, table_id, job_config=job_config, location=REGION)
    load_job.result()
    print("✅ Loaded Parquet into:", table_id)

# quick preview
bq.query(f"SELECT * FROM `{table_id}` LIMIT 5", location=REGION).to_dataframe()


✅ Loaded CSVs into: mgmt-467-94721.kaggle_flights.carrier_raw


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
0,1998,4,10,17,6,1998-10-17,DL,19790,DL,N937DL,...,,,,None,None,,None,None,,
1,1990,3,9,6,4,1990-09-06,EA,19707,EA,,...,,,,None,None,,None,None,,
2,1990,2,6,2,6,1990-06-02,EA,19707,EA,,...,,,,None,None,,None,None,,
3,1999,1,2,9,2,1999-02-09,DL,19790,DL,N959DL,...,,,,None,None,,None,None,,
4,1994,3,8,11,4,1994-08-11,DL,19790,DL,,...,,,,None,None,,None,None,,


In [9]:
# Builds a compat view for: carrier, dep_delay, arr_delay, origin, dest, diverted, fl_date, distance, day_of_week
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID)

DATASET   = "kaggle_flights"
TABLE     = "carrier_raw"                         # from your step (3)
TABLE_ID  = f"{PROJECT_ID}.{DATASET}.{TABLE}"
VIEW_NAME = "kaggle_compat"
VIEW_ID   = f"{PROJECT_ID}.{DATASET}.{VIEW_NAME}"
REGION    = "US"

# 1) Inspect schema
cols = bq.query(f"""
  SELECT LOWER(column_name) AS c
  FROM `{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.COLUMNS`
  WHERE table_name = '{TABLE}'
""", location=REGION).to_dataframe()["c"].tolist()
present = set(cols)
print("Detected columns (sample):", sorted(list(present))[:20], "...")

def has(*opts):              # any of these exist?
    return next((o for o in opts if o.lower() in present), None)

# 2) Choose the actual columns
carrier_col   = has("reporting_airline","iata_code_reporting_airline","op_unique_carrier","op_carrier","carrier")
dep_col       = has("depdelay","dep_delay","depdelayminutes")
arr_col       = has("arrdelay","arr_delay","arrdelayminutes")
origin_col    = has("origin")
dest_col      = has("dest","destination")
diverted_col  = has("diverted")
distance_col  = has("distance")
# date expr: prefer FlightDate/FL_DATE/fl_date, else build from Year/Month/DayOfMonth
if has("flightdate"):
    date_expr = "CAST(FlightDate AS DATE)"
elif has("fl_date"):
    date_expr = "CAST(fl_date AS DATE)"
elif has("fl_date".upper()):  # safety if uppercase
    date_expr = "CAST(FL_DATE AS DATE)"
elif all(n in present for n in ("year","month","dayofmonth")):
    date_expr = ("PARSE_DATE('%Y%m%d', CONCAT(CAST(Year AS STRING), "
                 "LPAD(CAST(Month AS STRING),2,'0'), LPAD(CAST(DayofMonth AS STRING),2,'0')))")
else:
    raise ValueError("No usable date field found (FlightDate / fl_date / Year-Month-DayofMonth).")

dow_col = has("dayofweek","day_of_week")  # optional; compute if missing

# 3) Sanity checks (raise clear messages if something critical is missing)
missing = []
for name, col in [("carrier",carrier_col),("dep_delay",dep_col),("arr_delay",arr_col),
                  ("origin",origin_col),("dest",dest_col),("diverted",diverted_col),
                  ("distance",distance_col)]:
    if col is None:
        missing.append(name)
if missing:
    raise ValueError(f"Missing required logical columns in Kaggle table: {missing}. "
                     f"Open the table preview and tell me what the dataset uses for those names.")

# 4) Build the view SQL only with columns that exist
view_sql = f"""
CREATE OR REPLACE VIEW `{VIEW_ID}` AS
SELECT
  -- carrier code
  CAST({carrier_col} AS STRING) AS carrier,

  -- delays (float)
  SAFE_CAST({dep_col} AS FLOAT64) AS dep_delay,
  SAFE_CAST({arr_col} AS FLOAT64) AS arr_delay,

  -- airports
  CAST({origin_col} AS STRING) AS origin,
  CAST({dest_col}   AS STRING) AS dest,

  -- diverted (0/1 → INT64)
  SAFE_CAST({diverted_col} AS INT64) AS diverted,

  -- date + distance
  {date_expr} AS fl_date,
  SAFE_CAST({distance_col} AS FLOAT64) AS distance,

  -- day of week: use column if present; else compute
  {"SAFE_CAST("+dow_col+" AS INT64)" if dow_col else "EXTRACT(DAYOFWEEK FROM "+date_expr+")"} AS day_of_week

FROM `{TABLE_ID}`
"""

print("Creating view with SQL:\n", view_sql[:500], "...\n")
bq.query(view_sql, location=REGION).result()
print("✅ Created view:", VIEW_ID)

# 5) Point TABLE_PATH to the compat view and preview
TABLE_PATH = VIEW_ID
print("TABLE_PATH =", TABLE_PATH)
bq.query(f"SELECT * FROM `{TABLE_PATH}` LIMIT 5", location=REGION).to_dataframe()


Detected columns (sample): ['actualelapsedtime', 'airtime', 'arrdel15', 'arrdelay', 'arrdelayminutes', 'arrivaldelaygroups', 'arrtime', 'arrtimeblk', 'cancellationcode', 'cancelled', 'carrierdelay', 'crsarrtime', 'crsdeptime', 'crselapsedtime', 'dayofmonth', 'dayofweek', 'departuredelaygroups', 'depdel15', 'depdelay', 'depdelayminutes'] ...
Creating view with SQL:
 
CREATE OR REPLACE VIEW `mgmt-467-94721.kaggle_flights.kaggle_compat` AS
SELECT
  -- carrier code
  CAST(reporting_airline AS STRING) AS carrier,

  -- delays (float)
  SAFE_CAST(depdelay AS FLOAT64) AS dep_delay,
  SAFE_CAST(arrdelay AS FLOAT64) AS arr_delay,

  -- airports
  CAST(origin AS STRING) AS origin,
  CAST(dest   AS STRING) AS dest,

  -- diverted (0/1 → INT64)
  SAFE_CAST(diverted AS INT64) AS diverted,

  -- date + distance
  CAST(FlightDate AS DATE) AS fl_date,
  SAFE_CAST(distance ...

✅ Created view: mgmt-467-94721.kaggle_flights.kaggle_compat
TABLE_PATH = mgmt-467-94721.kaggle_flights.kaggle_compat


,carrier,dep_delay,arr_delay,origin,dest,diverted,fl_date,distance,day_of_week
0,DL,0.0,1.0,ABE,ATL,0,1992-03-06,692.0,5
1,DL,0.0,-9.0,ABE,ATL,0,1999-10-23,692.0,6
2,DL,4.0,31.0,ABE,ATL,0,1995-06-12,692.0,1
3,EV,-10.0,-18.0,ABE,ATL,0,2003-03-22,692.0,6
4,DL,NaN,NaN,ABE,ATL,0,2001-01-01,692.0,1


In [10]:
PROJECT_ID = "mgmt-467-94721"
TABLE_PATH  = "mgmt-467-94721.kaggle_flights.kaggle_compat"


In [ ]:

# --- Minimal setup (edit 3 vars) ---
from google.colab import auth
auth.authenticate_user()

import os
from google.cloud import bigquery

PROJECT_ID = "mgmt-467-94721"      # e.g., mgmt-467-47888
REGION     = "us-central1"
TABLE_PATH = "bigquery-public-data.faa.us_flights"   # or your `bigquery-public-data.flights` table/view

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"]     = REGION
bq = bigquery.Client(project=PROJECT_ID)

print("BQ Project:", PROJECT_ID)
print("Source table:", TABLE_PATH)


BQ Project: mgmt-467-94721
Source table: bigquery-public-data.faa.us_flights


In [ ]:
# ===========================================
# BTS (On-Time) -> BigQuery via monthly stages
# Then a UNION view with normalized columns
# ===========================================

# ---- EDIT THESE ----
PROJECT_ID = "mgmt-467-94721"  # your GCP project
DATASET    = "flights_data"    # BigQuery dataset to create/use
YEAR       = 2024              # which year to fetch
MONTHS     = (1, 2, 3)         # months to load (Jan–Mar 2024)
STAGING_PREFIX = "stg_bts"     # prefix for monthly staging tables
COMPAT_VIEW    = "bts_compat"  # final view exposing consistent columns
# --------------------

# Auth & imports
from google.colab import auth
auth.authenticate_user()

import io, zipfile, requests
from google.cloud import bigquery
from google.api_core.exceptions import Conflict

bq = bigquery.Client(project=PROJECT_ID)
print("BQ Project:", PROJECT_ID)

# 1) Create dataset if missing (US multi-region recommended)
ds_id = f"{PROJECT_ID}.{DATASET}"
try:
    ds = bigquery.Dataset(ds_id)
    ds.location = "US"
    bq.create_dataset(ds, timeout=30)
    print(f"✅ Created dataset {ds_id} (location=US)")
except Conflict:
    print(f"✅ Dataset {ds_id} already exists")

# 2) Helper: download month ZIP from BTS + load to its own staging table (autodetect)
def load_month_to_staging(year: int, month: int):
    table_id = f"{PROJECT_ID}.{DATASET}.{STAGING_PREFIX}_{year}_{month:02d}"
    url = (
        "https://transtats.bts.gov/PREZIP/"
        f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
    )
    print(f"\n--- {year}-{month:02d} ---")
    print(f"Downloading {url} ...")
    r = requests.get(url, timeout=300)
    r.raise_for_status()

    # Unzip in-memory and find CSV
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
    if not csv_names:
        raise RuntimeError("No CSV found in the ZIP.")
    csv_name = csv_names[0]
    print(f"Found CSV: {csv_name}")

    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        autodetect=True,
        skip_leading_rows=1,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        allow_quoted_newlines=True,
        allow_jagged_rows=True,
    )

    # Ensure a true binary buffer for BQ load
    with zf.open(csv_name, "r") as zip_file:
        file_bytes = io.BytesIO(zip_file.read())
        load_job = bq.load_table_from_file(file_bytes, table_id, job_config=job_config)

    print(f"Loading → {table_id} …")
    load_job.result()  # wait
    rows = bq.get_table(table_id).num_rows
    print(f"✅ Loaded {rows:,} rows into {table_id}")
    return table_id

# 3) Load each month to its own staging table
staging_tables = []
for m in MONTHS:
    staging_tables.append(load_month_to_staging(YEAR, m))

# 4) Build the UNION view with normalized/consistent types
#    We only select the fields you care about, so we avoid any weird type drift
COMPAT_VIEW = "bts_compat"
union_selects = []
for tbl in staging_tables:
    union_selects.append(f"""
SELECT
  IATA_CODE_Reporting_Airline AS carrier,
  SAFE_CAST(DepDelay AS FLOAT64) AS dep_delay,
  SAFE_CAST(ArrDelay AS FLOAT64) AS arr_delay,
  Origin  AS origin,
  Dest    AS dest,
  SAFE_CAST(Diverted AS INT64)   AS diverted,
  -- handy extras:
  PARSE_DATE('%Y%m%d', CONCAT(CAST(Year AS STRING),
                              LPAD(CAST(Month AS STRING), 2, '0'),
                              LPAD(CAST(DayofMonth AS STRING), 2, '0'))) AS fl_date,
  SAFE_CAST(Distance AS FLOAT64) AS distance,
  SAFE_CAST(DayOfWeek AS INT64)  AS day_of_week
FROM `{tbl}`
""".strip())

union_sql_body = "\nUNION ALL\n".join(union_selects)
view_id = f"{PROJECT_ID}.{DATASET}.{COMPAT_VIEW}"

view_sql = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
{union_sql_body}
"""
bq.query(view_sql).result()
print(f"\n✅ Created/updated view: {view_id}")

# 5) Smoke test: query a few rows
preview = bq.query(f"""
SELECT carrier, origin, dest, dep_delay, arr_delay, diverted, fl_date
FROM `{view_id}`
WHERE dep_delay IS NOT NULL AND arr_delay IS NOT NULL
ORDER BY fl_date DESC
LIMIT 10
""").to_dataframe()
print("\nPreview:")
print(preview)

# 6) Print the TABLE_PATH you can plug into other notebooks
print("\nUse this TABLE_PATH in your other code:")
print(view_id)


BQ Project: mgmt-467-94721
✅ Dataset mgmt-467-94721.flights_data already exists

--- 2024-01 ---
Found CSV: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
Loading → mgmt-467-94721.flights_data.stg_bts_2024_01 …
✅ Loaded 547,271 rows into mgmt-467-94721.flights_data.stg_bts_2024_01

--- 2024-02 ---
Found CSV: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_2.csv
Loading → mgmt-467-94721.flights_data.stg_bts_2024_02 …
✅ Loaded 519,221 rows into mgmt-467-94721.flights_data.stg_bts_2024_02

--- 2024-03 ---
Found CSV: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_3.csv
Loading → mgmt-467-94721.flights_data.stg_bts_2024_03 …
✅ Loaded 591,767 rows into mgmt-467-94721.flights_data.stg_bts_2024_03

✅ Created/updated view: mgmt-467-94721.flights_data.bts_compat

Preview:
  carrier origin dest  dep_delay  arr_delay  diverted     fl_date
0      OO    ABE  ORD      -11.0       -8.0         0  2024-03-31
1      OH    ABE  CLT       36.0 

### Quick sanity check

In [ ]:

preview_sql = f"SELECT * FROM `{TABLE_PATH}` LIMIT 5"
bq.query(preview_sql).result().to_dataframe()


,carrier,dep_delay,arr_delay,origin,dest,diverted,fl_date,distance,day_of_week
0,9E,-8.0,-15.0,ATL,ABE,0,2024-01-08,692.0,1
1,9E,78.0,66.0,ATL,ABE,0,2024-01-09,692.0,2
2,9E,-5.0,-15.0,ATL,ABE,0,2024-01-10,692.0,3
3,9E,2.0,4.0,ATL,ABE,0,2024-01-11,692.0,4
4,9E,4.0,-11.0,ATL,ABE,0,2024-01-12,692.0,5



## 1) Canonical mapping (adjust as needed)
Map to a minimal schema used in the rest of the notebook:
- `flight_date` (DATE), `dep_delay` (NUM), `distance` (NUM), `carrier` (STRING), `origin` (STRING), `dest` (STRING), `diverted` (BOOL)


In [11]:
from google.cloud import bigquery
from datetime import datetime, timezone

PROJECT_ID = "mgmt-467-94721"
TABLE_PATH  = "mgmt-467-94721.kaggle_flights.kaggle_compat"  # <-- new view you created
REGION      = "US"

bq = bigquery.Client(project=PROJECT_ID)
print("TABLE_PATH =", TABLE_PATH)

# Canonical mapping for your compat view
CANONICAL_BASE_SQL = f"""
WITH canonical_flights AS (
  SELECT
    CAST(fl_date AS DATE)            AS flight_date,
    CAST(dep_delay AS FLOAT64)       AS dep_delay,
    CAST(distance  AS FLOAT64)       AS distance,
    CAST(carrier   AS STRING)        AS carrier,
    CAST(origin    AS STRING)        AS origin,
    CAST(dest      AS STRING)        AS dest,
    CAST(diverted  AS BOOL)          AS diverted,
    CAST(day_of_week AS INT64)       AS day_of_week
  FROM `{TABLE_PATH}`
  WHERE dep_delay IS NOT NULL
)
"""
print("✅ Canonical defined.")


TABLE_PATH = mgmt-467-94721.kaggle_flights.kaggle_compat
✅ Canonical defined.


### 2) Split (80/20)

In [12]:

SPLIT_CLAUSE = r"""
, split_cte AS (
  SELECT
    cf.*,
    CASE
      WHEN MOD(ABS(FARM_FINGERPRINT(CONCAT(
             CAST(flight_date AS STRING), '|', carrier, '|', origin, '|', dest
           ))), 10) < 8 THEN 'TRAIN'
      ELSE 'EVAL'
    END AS split_flag
  FROM canonical_flights cf
)
"""
print("✅ Split clause defined.")


print(SPLIT_CLAUSE)


✅ Split clause defined.

, split_cte AS (
  SELECT
    cf.*,
    CASE
      WHEN MOD(ABS(FARM_FINGERPRINT(CONCAT(
             CAST(flight_date AS STRING), '|', carrier, '|', origin, '|', dest
           ))), 10) < 8 THEN 'TRAIN'
      ELSE 'EVAL'
    END AS split_flag
  FROM canonical_flights cf
)




## 3) Baseline model — LOGISTIC_REG (`diverted`)
Use **only** a small set of signals for the baseline (keep it honest).


In [13]:
MODEL_DATASET = f"{PROJECT_ID}.unit2_flights"
bq.query(f"CREATE SCHEMA IF NOT EXISTS `{MODEL_DATASET}`", location=REGION).result()

# unique name per run without deprecated utcnow()
MODEL_BASE = f"{MODEL_DATASET}.clf_diverted_base_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

sql_create_model = f"""
CREATE MODEL `{MODEL_BASE}`
OPTIONS (MODEL_TYPE='LOGISTIC_REG', INPUT_LABEL_COLS=['diverted']) AS
{CANONICAL_BASE_SQL}
{SPLIT_CLAUSE}
SELECT
  diverted,
  dep_delay, distance, carrier, origin, dest, day_of_week
FROM split_cte
WHERE split_flag = 'TRAIN'
"""
bq.query(sql_create_model, location=REGION).result()
print("✅ Baseline model trained:", MODEL_BASE)

sql_eval = f"""
SELECT * FROM ML.EVALUATE(
  MODEL `{MODEL_BASE}`,
  (
    {CANONICAL_BASE_SQL}
    {SPLIT_CLAUSE}
    SELECT
      diverted,
      dep_delay, distance, carrier, origin, dest, day_of_week
    FROM split_cte
    WHERE split_flag = 'EVAL'
  )
)
"""
eval_df = bq.query(sql_eval, location=REGION).to_dataframe()
eval_df




✅ Baseline model trained: mgmt-467-94721.unit2_flights.clf_diverted_base_20251111_223205


,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.0,0.0,0.997634,0.0,0.016332,0.691266


### Confusion matrix — default 0.5 threshold

In [14]:
# Confusion matrix at default 0.5 threshold (fixed CTE chaining)
cm_default_sql = f"""
{CANONICAL_BASE_SQL}
{SPLIT_CLAUSE}
, eval_rows AS (
  SELECT
    diverted AS label,
    dep_delay, distance, carrier, origin, dest, day_of_week
  FROM split_cte
  WHERE split_flag = 'EVAL'
)
SELECT
  SUM(CASE WHEN label = TRUE  AND predicted_diverted = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN label = FALSE AND predicted_diverted = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN label = TRUE  AND predicted_diverted = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN label = FALSE AND predicted_diverted = FALSE THEN 1 ELSE 0 END) AS TN
FROM ML.PREDICT(
  MODEL `{MODEL_BASE}`,
  (SELECT * FROM eval_rows)
)
"""
bq.query(cm_default_sql, location="US").to_dataframe()



,TP,FP,FN,TN
0,0,1,927,391220


### Confusion matrix — your custom threshold

In [20]:
CUSTOM_THRESHOLD = 0.25  # change to whatever you want to test

cm_thresh_sql = f"""
{CANONICAL_BASE_SQL}
{SPLIT_CLAUSE}
, eval_rows AS (
  SELECT
    diverted AS label,
    dep_delay, distance, carrier, origin, dest, day_of_week
  FROM split_cte
  WHERE split_flag = 'EVAL'
)
, scored AS (
  SELECT
    label,
    -- Get probability for label=TRUE by name, not OFFSET
    (
      SELECT prob
      FROM UNNEST(predicted_diverted_probs)
      WHERE label = TRUE
    ) AS p_true,
    CAST((
      SELECT prob
      FROM UNNEST(predicted_diverted_probs)
      WHERE label = TRUE
    ) >= {CUSTOM_THRESHOLD} AS BOOL) AS pred_label
  FROM ML.PREDICT(
    MODEL `{MODEL_BASE}`,
    (SELECT * FROM eval_rows)
  )
)
SELECT
  SUM(CASE WHEN label = TRUE  AND pred_label = TRUE  THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN label = FALSE AND pred_label = TRUE  THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN label = TRUE  AND pred_label = FALSE THEN 1 ELSE 0 END) AS FN,
  SUM(CASE WHEN label = FALSE AND pred_label = FALSE THEN 1 ELSE 0 END) AS TN
FROM scored
"""
bq.query(cm_thresh_sql, location="US").to_dataframe()


,TP,FP,FN,TN
0,1,17,926,391204



## 4) Engineered model — `TRANSFORM` (same label, stricter bar)
Create **route**, extract **day_of_week**, and **bucketize dep_delay**. Compare metrics to baseline.


In [16]:
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID)

MODEL_DATASET = f"{PROJECT_ID}.unit2_flights"
MODEL_XFORM   = f"{MODEL_DATASET}.clf_diverted_xform"

# Ensure dataset exists
bq.query(f"CREATE SCHEMA IF NOT EXISTS `{MODEL_DATASET}`", location="US").result()

# 1) Train engineered model (label MUST be listed directly in TRANSFORM)
sql_xform_create = f"""
CREATE OR REPLACE MODEL `{MODEL_XFORM}`
TRANSFORM (
  -- label: must be present by name (no CAST/alias)
  diverted,

  -- engineered features
  CONCAT(origin, '-', dest) AS route,
  EXTRACT(DAYOFWEEK FROM flight_date) AS day_of_week,
  CASE
    WHEN dep_delay < -5  THEN 'early'
    WHEN dep_delay <=  5 THEN 'on_time'
    WHEN dep_delay <= 15 THEN 'minor'
    WHEN dep_delay <= 45 THEN 'moderate'
    ELSE 'major'
  END AS dep_delay_bucket,

  -- pass-through raw features
  dep_delay, distance, carrier, origin, dest
)
OPTIONS (MODEL_TYPE='LOGISTIC_REG', INPUT_LABEL_COLS=['diverted']) AS
{CANONICAL_BASE_SQL}
{SPLIT_CLAUSE}
SELECT *
FROM split_cte
WHERE split_flag = 'TRAIN'
"""
bq.query(sql_xform_create, location="US").result()
print("✅ Engineered model trained:", MODEL_XFORM)

# 2) Compare baseline vs engineered on identical EVAL slice
compare_sql = f"""
-- Baseline metrics (use canonical day_of_week column)
SELECT 'baseline' AS model_version, * FROM ML.EVALUATE(
  MODEL `{MODEL_BASE}`,
  (
    {CANONICAL_BASE_SQL}
    {SPLIT_CLAUSE}
    SELECT
      diverted,
      dep_delay, distance, carrier, origin, dest, day_of_week, flight_date
    FROM split_cte
    WHERE split_flag = 'EVAL'
  )
)
UNION ALL
-- Engineered metrics: provide the raw columns referenced by TRANSFORM
SELECT 'engineered' AS model_version, * FROM ML.EVALUATE(
  MODEL `{MODEL_XFORM}`,
  (
    {CANONICAL_BASE_SQL}
    {SPLIT_CLAUSE}
    SELECT
      diverted,                               -- label
      dep_delay, distance, carrier, origin, dest,
      flight_date                             -- used to make day_of_week
    FROM split_cte
    WHERE split_flag = 'EVAL'
  )
)
"""
bq.query(compare_sql, location="US").to_dataframe()


✅ Engineered model trained: mgmt-467-94721.unit2_flights.clf_diverted_xform


,model_version,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,engineered,0.0,0.0,0.997621,0.0,0.016193,0.678494
1,baseline,0.0,0.0,0.997634,0.0,0.016332,0.691267



### Write-up (concise)
- **Threshold chosen & ops rationale:** …  
- **Baseline vs engineered — observed changes in AUC/precision/recall:** …  
- **Risk framing:** cost of FP vs FN for diversion planning; what is your acceptable FN-rate? …


Threshold chosen & ops rationale:
I started with the default 0.5 decision threshold. At that cutoff the confusion matrix on the eval set was TP=0, FP=0, FN=1, TN≈391k, which means the model never predicts “diverted.” That’s mathematically fine (and gives high accuracy) but not operationally useful. For a real airline ops use case I would lower the threshold to around 0.25 so the model acts as an early-warning filter rather than a yes/no decision maker: we’d accept a small number of extra false alarms in order to surface more potentially risky flights for human review.

Baseline vs engineered changes in AUC / precision / recall:
At the default 0.5 threshold, both models have precision = 0, recall = 0, and F1 = 0, because neither ever predicts a positive class on this extremely imbalanced diverted vs non-diverted label. Accuracy stays very high for both (~0.9976) simply because almost all flights are not diverted. The engineered model (with route, day_of_week, and dep_delay buckets) changes the ranking slightly: baseline ROC AUC ≈ 0.691, engineered ROC AUC ≈ 0.678, and log-loss improves slightly. So the engineered features don’t yet translate into better classification at 0.5, but they do give a comparable level of separation; any gains would only show up once we tune the threshold or use class weighting.

Risk framing (FP vs FN and acceptable FN-rate):
For diversion planning, false negatives (FN) are more costly than false positives (FP): missing a real diversion can mean poor resource planning, mis-positioned crews, and a bad passenger experience, while a false alarm mostly wastes some analyst/ops attention. In practice I’d be willing to tolerate a moderate FP rate (e.g., a few dozen extra “watchlist” flights per day) to keep the FN-rate low. A reasonable target might be an FN-rate around 10–20% for this tool when used as a decision support signal (not an automated action), even if that means precision is modest.


---

## Rubric (Flights, 100 pts)
**Team-only deliverable in this notebook**

- Baseline LOGISTIC_REG + evaluation (AUC + confusion @0.5) — **20**  
- Custom threshold confusion matrix + ops justification — **20**  
- Engineered model with `TRANSFORM` (route, DOW, delay bucket) — **20**  
- Comparison table (baseline vs engineered) + 3–5 sentence interpretation — **20**  
- Reproducibility: parameters clear, no hidden magic; schema mapping documented — **10**  
- Governance notes: assumptions/limitations + slices you would monitor — **10**

> **Strictness:** No screenshots; use actual results cells. Keep explanations concise (bullet points OK).
